# Week 3 · Day 9 — Introduction to AI APIs & Large Language Models

**Course:** IPAM USL 5-Week Short Course: Introduction to Artificial Intelligence *(Introductory tier)*

**Facilitator:** Solomon Wilson MBCS | PhD Student, Computer Science | Deputy HOD Transport Planning & Operations | HOD, IT & Audit, SLPTA

**Mode:** Google Colab (zero-install)

**Mental model layer:** L9 — Calling an LLM through an API

**Running scenario:** Route **R12** (Wilberforce → CBD) — operator OP-104, 25-minute delay

**Module:** 2 · **Week:** 4 · **Tier:** Intro

**New concept:** LLMs predict the next token — they complete patterns, not reason from facts

**Deliverable wired in:** None

## Learning objectives
By the end of today you will be able to:
- Explain at a high level how an **LLM** works (tokens, prompts) and what an **API** is.
- Make your first calls to **Google Gemini** from Colab.
- Use an LLM to **summarise**, **classify**, and **extract** information from SLPTA text.

## Why this matters for SLPTA

Until now we worked with numbers and images. But dispatch runs on **text** — incident reports, passenger complaints, policy documents. A large language model can read an R12 incident report and summarise it, sort a complaint into a category, or pull the key facts into a tidy record — in seconds. Today we make that connection for the first time.

## Environment setup
Today we genuinely need the Gemini client, so the install matters.

In [ ]:
# google-genai is REQUIRED today — it is how we talk to Gemini.
!pip install -q google-genai
print("Environment ready.")

Environment ready.


In [ ]:
# --- Standard SLPTA bootstrap (identical in every notebook) ----------------
import sys
from pathlib import Path
for candidate in [Path.cwd(), *Path.cwd().parents,
                  Path("/content/IPAM_USL_Intro_AI_5Week")]:
    if (candidate / "shared" / "slpta_bootstrap.py").exists():
        sys.path.insert(0, str(candidate / "shared"))
        break

from slpta_bootstrap import (MODEL, ensure_course_data, get_client,
                             load_route12_context, load_route_logs,
                             load_complaints, load_routes, load_operators)

ensure_course_data()
print("Model configured:", MODEL)
print(load_route12_context())

Model configured: gemini-3.5-flash
Route R12 (Wilberforce → CBD). The 07:45 service, operated by OP-104 on vehicle BUS-123, departed 25 minutes late. Recorded cause: Heavy traffic on Wilkinson Road. About 40 passengers were affected and the dispatch desk received multiple complaints. (Synthetic SLPTA scenario — no real data.)


## One-time setup — your API key (required today)

1. Get a free Gemini API key from **aistudio.google.com** → *Get API key*.
2. In Colab, click the **key icon** (Secrets) in the left sidebar.
3. Add a secret named **`GEMINI_API_KEY`**, paste your key, and toggle **notebook access** on.
4. Run the next cell. If it complains about a missing key, repeat step 3 and re-run.

<!-- cell-diagram:c08 -->
<p align="center"></p>

### Check your understanding (before running)
We are about to make our very first call to the Gemini API. The test question asks Gemini to explain LLMs and APIs in one sentence.

**Predict:** Will Gemini mention tokens? Will it explain what an API is? How long will the answer be?

*Write your prediction, then run.*

In [ ]:
import google.genai as genai
from google.genai import errors

# --- Gemini API Configuration ---
client = get_client()
gemini_model = "gemini-2.5-flash"

def ask(prompt):
    """
    Sends a text-based prompt to Gemini with error handling for quota limits.
    """
    try:
        # generate_content uses the global MODEL variable defined during bootstrap
        response = client.models.generate_content(model=gemini_model, contents=prompt)
        return response.text
    except errors.ClientError as e:
        if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
            return "[ERROR: API Quota Exceeded. Please wait a minute or check your Google AI Studio limits.]"
        else:
            return f"[ERROR: {str(e)}]"

print("Connected successfully. Testing the connection...")

# Initial test call to confirm API connectivity
test_query = "In one short sentence, explain how an LLM works and what an API is?"
print(f"Q: {test_query}")
print(f"A: {ask(test_query)}")

Connected successfully. Testing the connection...
Q: In one short sentence, explain how an LLM works and what an API is?
A: An LLM generates human-like text by predicting the next most probable word based on patterns learned from massive data, and an API is a standardized interface for other programs to request and receive its output.


> **If you see a 429 error:** The free Gemini tier has a per-minute request limit.
> Wait 60 seconds and re-run the cell. This is normal — it means the API is working,
> just rate-limited. Your API key is fine.

## Concept — first principles

A **large language model (LLM)** is trained to predict the next chunk of text (a **token**) over and over, which lets it write fluent replies. It does **not** look anything up — it predicts plausible text. So if we want answers grounded in SLPTA facts, we must **put those facts in the prompt**.

An **API** is simply a way for our code to send a request (our prompt) to the model and get a response back.

*Jargon, defined once:* a **token** is a piece of a word the model reads and writes; a **prompt** is the text instruction we send.

<p align="center"></p>

### Check your understanding (before running)
We will ask Gemini to summarise the real R12 incident report. **Predict:** will it stick to the facts in the report, or might it add details that were never written down?

In [ ]:
# --- Data Loading & Preparation ---

# ensure_course_data() downloads/verifies the SLPTA dataset and returns the Path object
data_path = ensure_course_data()

# Construct the absolute path to the specific incident report for Route R12
# We use the / operator from pathlib for clean, cross-platform path joining
incident_file_path = data_path / "incident_reports" / "incident_R12_2025-02-14.txt"

# Read the raw text content of the report to be used as 'grounding' context for the LLM
try:
    incident = incident_file_path.read_text()
    print("--- Incident Report Loaded ---")
    print(incident)
except FileNotFoundError:
    print(f"Error: The file at {incident_file_path} was not found.")
    incident = ""

--- Incident Report Loaded ---
On 14 Feb 2025 the 07:45 R12 service (Wilberforce to CBD), vehicle SLPTA-1142 under operator OP-104, departed 25 minutes late. The operator reported heavy traffic on Wilkinson Road following a stalled truck. About 40 passengers were affected. Dispatch logged the cause as Traffic and issued a passenger advisory at 08:02. No injuries. Follow-up: review peak headway on the Wilberforce corridor.


<!-- cell-diagram:c14 -->
<p align="center"></p>

### Check your understanding (before running)
Gemini will summarise the pasted R12 incident report in a few sentences.

**Predict:** Will the summary mention the **25-minute delay** and **OP-104** without making up facts?

In [ ]:
# DEMO 1 — Summarise. We GROUND the model by pasting the report into the prompt.
summary = ask(
    "Summarise this SLPTA incident report in 2 sentences for a manager. "
    "Use only facts stated in the report.\n\n" + incident
)
print(summary)

On February 14, 2025, the 07:45 R12 service (SLPTA-1142) departed 25 minutes late, affecting approximately 40 passengers due to heavy traffic on Wilkinson Road caused by a stalled truck. Dispatch logged the incident, issued a passenger advisory at 08:02, and recommended a follow-up review of peak headway on the Wilberforce corridor.


<!-- cell-diagram:c16 -->
<p align="center"></p>

### Check your understanding (before running)
We will ask Gemini to classify a random SLPTA passenger complaint into one of 7 categories.

**Predict:** Without seeing the complaint text, which category do you think will appear most often in the Route R12 data — Delay, Safety, or Overcharging? Why?

*Write your prediction, then run.*

In [ ]:
# DEMO 2 — Classify a passenger complaint into a category (zero-shot).
complaint = load_complaints().sample(1, random_state=5)["raw_text"].iloc[0]
print("Complaint:", complaint, "\n")

category = ask(
    "Classify this SLPTA passenger complaint into exactly ONE category from: "
    "Delay, Overcharging, Safety, Cleanliness, Staff Conduct, Lost Item, Other. "
    "Reply with only the category name.\n\nComplaint: " + complaint
)
print("Predicted category:", category.strip())

Complaint: Lost a bag of documents on R7, who do I contact? 

Predicted category: Lost Item


<!-- cell-diagram:c18 -->
<p align="center"></p>

### Check your understanding (before running)
We will ask Gemini to extract structured fields (route, operator, delay_minutes, cause) from the R12 incident report.

**Predict:** Will the model get all four fields correct? What do you think it will write for `cause`? Will it invent any details not in the report?

*Write your prediction, then run.*

In [ ]:
# DEMO 3 — Extract structured fields from the unstructured report.
fields = ask(
    "From the incident report below, extract these fields as simple lines "
    "(route, operator, delay_minutes, cause). If a field is not stated, write "
    "'unknown'. Use only the report.\n\n" + incident
)
print(fields)

route: R12 (Wilberforce to CBD)
operator: OP-104
delay_minutes: 25
cause: Traffic


### Exercise — change the task

Replace `___FILL_TASK___` below with your own instruction — for example *"Draft a one-line apology SMS to passengers, under 160 characters."* Then run it.

<!-- cell-diagram:c21 -->
<p align="center"></p>

### Check your understanding (before running)
You are about to write your own instruction for Gemini to apply to the R12 incident report.

**Before you fill in `my_task`:** Write one sentence describing what you want to ask — for example, a passenger apology SMS, a manager summary, or a follow-up action list. Then fill in `my_task` and predict what Gemini will produce.

*Write your prediction, then run.*

In [ ]:
# --- Exercise: Custom LLM Instruction ---

# User Task: Replace the placeholder string below with a specific instruction.
# Examples: "Draft an apology SMS under 160 characters" or "Extract the vehicle ID."
my_task = "___FILL_TASK___"

# Guardrail: Ensure the user has actually provided a task before calling the API.
# This prevents unnecessary API calls and avoids sending placeholder text to the model.
if "___FILL_TASK___" in my_task:
    print("Action Required: Please edit 'my_task' above with your own instruction, then re-run this cell.")
else:
    # Construct the final prompt by combining the user's task with the grounded context (incident report).
    # We use double newlines to clearly separate the instruction from the data.
    full_prompt = f"{my_task}\n\nIncident report:\n{incident}"

    # Send the prompt to Gemini via the 'ask' helper function and print the result.
    print(f"Executing Task: {my_task}\n")
    print(ask(full_prompt))

## A word on responsible use

The model predicts plausible text — it can sound confident and still be wrong (a **hallucination**). That is why every prompt above **pasted the real report** and said *"use only the facts stated."* For anything that affects passengers or money, treat the output as a draft a human must check.

## Check your understanding
1. In one sentence, what does an LLM actually do under the hood?
2. Why did we paste the incident report into the prompt instead of just asking "what happened to R12"?
3. Name one SLPTA task where you would **not** act on the model's answer without a human checking it first.

### Your answers
*Double-click to edit this cell and type your answers here.*

1.
2.
3.

## If you remember one thing today…

> **An LLM predicts text from your prompt — give it the facts to work from, and always check before acting.**

## Submission checklist
- [ ] Run every code cell successfully (top to bottom)
- [ ] Complete the exercise (fill every blank / make the requested change)
- [ ] Answer the **Check your understanding** questions in the markdown cell provided
- [ ] Save a clean copy of the notebook (*File → Save a copy in Drive*)